# Battleshots fraud prediction
Load the saved model and score the supplied transactions in their original order.


## Saved model and validation

This prediction-only notebook loads the saved logistic regression (C=0.3), caps recent 24-hour spend at the labelled-training 99th percentile (4926.9294), then adds a night-by-new-device interaction to the prior 31 features. It never uses transaction ID or the fraud label as an input. The model was fitted on all 20,000 labelled rows before export.

The interaction improved average precision on five stratified five-fold split seeds by about 0.0095 on average and on two additional holdouts. The exploratory public CSV #3965 scored 0.19095, versus the prior selected CSV #3683 at 0.18568. The preset 0.010 mean-gain gates were narrowly missed, so these results are uncertain and the hidden private score is unknown. The notebook only loads the saved coefficients and writes one probability per supplied row; it has no training cells.


In [ ]:
"""Feature engineering for Track 2. Pure function of one row set: the same code runs at training and prediction."""
import numpy as np
import pandas as pd

CATEGORICAL = ["merchant_category", "country", "transaction_channel"]
NUMERIC_RAW = ["transaction_amount", "transaction_hour", "transactions_last_24h", "spend_last_24h",
               "account_age", "new_device", "transactions_last_1h"]
HIGH_RISK_CATEGORIES = {"luxury", "cash_transfer", "electronics", "travel"}


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for c in NUMERIC_RAW:
        out[c] = pd.to_numeric(df[c], errors="coerce")
    # Was the value missing? Missing fields are themselves a weak signal and are far more common in the test set.
    for c in NUMERIC_RAW + CATEGORICAL:
        out[f"{c}_missing"] = df[c].isna().astype(int)
    amt = out["transaction_amount"]
    out["amount_log"] = np.log1p(amt)
    # How this payment compares with the account's recent behaviour.
    avg_recent = out["spend_last_24h"] / out["transactions_last_24h"].replace(0, np.nan)
    out["amount_vs_recent_avg"] = amt / avg_recent.replace(0, np.nan)
    out["amount_share_of_24h"] = amt / (out["spend_last_24h"] + amt)
    out["burst_1h_share"] = out["transactions_last_1h"] / out["transactions_last_24h"].replace(0, np.nan)
    out["is_night"] = out["transaction_hour"].isin([23, 0, 1, 2, 3, 4]).astype(int)
    out["account_age_log"] = np.log1p(out["account_age"])
    out["is_new_account"] = (out["account_age"] < 90).astype(int)
    out["is_foreign"] = (df["country"].notna() & (df["country"] != "SG")).astype(int)
    out["high_risk_category"] = df["merchant_category"].isin(HIGH_RISK_CATEGORIES).astype(int)
    out["new_device_foreign"] = out["new_device"].fillna(0).astype(int) * out["is_foreign"]
    out["new_device_high_risk"] = out["new_device"].fillna(0).astype(int) * out["high_risk_category"]
    for c in CATEGORICAL:
        out[c] = df[c].fillna("missing").astype("category")
    out["night_new_device"] = out["is_night"] * out["new_device"]
    return out


def feature_names(df: pd.DataFrame) -> list[str]:
    return list(make_features(df.head(1)).columns)


In [ ]:
def score_exported(artifact, raw):
    """Reproduce the fitted sklearn pipeline using only NumPy and pandas."""
    if artifact.get("schema_version") != 1:
        raise ValueError("Unsupported model artifact")
    if artifact.get("feature_mode") != "base_night_device":
        raise ValueError("Unsupported feature mode: expected base_night_device")
    features = make_features(raw)
    if list(features.columns) != artifact["feature_names"]:
        raise ValueError("Saved feature schema does not match input")
    numeric = artifact["numeric_names"]
    categorical = artifact["categorical_names"]
    categories = artifact["categories"]
    coef = np.asarray(artifact["coefficients"], dtype=float)
    if len(coef) != len(numeric) + sum(map(len, categories)) or len(categorical) != len(categories):
        raise ValueError("Corrupt model coefficients")

    values = features[numeric].to_numpy(dtype=float)
    if np.isinf(values).any():
        raise ValueError("Infinite numeric feature")
    medians = np.asarray(artifact["imputer_medians"], dtype=float)
    means = np.asarray(artifact["scaler_means"], dtype=float)
    scales = np.asarray(artifact["scaler_scales"], dtype=float)
    if not (len(numeric) == len(medians) == len(means) == len(scales)) or (scales <= 0).any():
        raise ValueError("Corrupt numeric preprocessing")
    values = np.where(np.isnan(values), medians, values)
    logits = ((values - means) / scales) @ coef[:len(numeric)] + artifact["intercept"]

    offset = len(numeric)
    for name, known in zip(categorical, categories):
        weight = dict(zip(known, coef[offset:offset + len(known)]))
        logits += features[name].astype("string").map(weight).fillna(0).to_numpy(dtype=float)
        offset += len(known)
    scores = np.exp(-np.logaddexp(0, -logits))
    if len(scores) != len(raw) or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Invalid model probabilities")
    return scores


In [ ]:
import os
from pathlib import Path
import json

input_path = Path(os.environ.get("DATATHON_INPUT_PATH", "test.csv"))
output_path = Path(os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv"))
raw = pd.read_csv(input_path)
required = set(NUMERIC_RAW + CATEGORICAL)
if raw.empty or raw.columns.duplicated().any() or not required.issubset(raw.columns):
    raise ValueError("Invalid prediction input schema")
artifact = json.loads(Path("model.json").read_text())
raw["spend_last_24h"] = pd.to_numeric(raw["spend_last_24h"], errors="coerce").clip(upper=4926.929399999964)
scores = score_exported(artifact, raw)
pd.DataFrame({"prediction": scores}).to_csv(output_path, index=False)
print(f"Wrote {len(scores)} probabilities to {output_path}")
